In [ ]:
# %% [Cell: V7 vs V4 Elite Specialist Audit]
from sklearn.metrics import mean_absolute_error
from IPython.display import Markdown, display
import numpy as np
import pandas as pd

# 1. Dynamically find elite players inside the V7 validation set
# (This bypasses any older variables and prevents the IndexingError)
player_season_avgs = df_augmented.groupby('Player UUID')['Total Points'].mean()
elite_uuids = player_season_avgs[player_season_avgs > 4.5].index

# Map the V7 validation windows to their Player UUIDs
val_uuids = df_augmented.loc[X_val_v7.index, 'Player UUID']

# Create a clean numpy boolean mask of just the elite players
elite_val_mask_v7 = val_uuids.isin(elite_uuids).values

# 2. Filter the features and target using the clean mask
X_val_v7_elites = X_val_v7[elite_val_mask_v7].copy()
y_val_elites    = y_val_v7[elite_val_mask_v7].copy()

# Pull the metadata for the final table
meta_elites = df_augmented.loc[X_val_v7_elites.index, ['Web Name', 'season', 'Gameweek', 'Rolling_xGI_5']]

# 3. Align V4 features perfectly
# V4 uses 12 features (everything except 'rolling_xgi'). 
# We pull them directly from the V7 set to guarantee 100% alignment.
v4_features = [c for c in X_val_v7.columns if c != 'rolling_xgi']
X_val_v4_elites = X_val_v7_elites[v4_features].copy()

# 4. Generate Predictions
preds_v4 = model_v4.predict(X_val_v4_elites)
preds_v7 = model_v7.predict(X_val_v7_elites)

mae_elite_v4 = mean_absolute_error(y_val_elites, preds_v4)
mae_elite_v7 = mean_absolute_error(y_val_elites, preds_v7)

v7_truth_df = pd.DataFrame({
    'Player': meta_elites['Web Name'].values,
    'GW': meta_elites['Gameweek'].values,
    'xGI (Form)': np.round(meta_elites['Rolling_xGI_5'].values, 2),
    'Actual Points': y_val_elites.values,
    'V4_Predicted': np.round(preds_v4, 2),
    'V7_Predicted': np.round(preds_v7, 2),
    'V7_Error': np.round(abs(preds_v7 - y_val_elites.values), 2)
})

# 5. Display Results
report = f"""
# V7 Breakthrough Analysis: The Power of xG/xA

### Accuracy on Elite Players (>4.5 Avg)
| Model Version | Elite MAE (Lower is Better) |
| :--- | :--- |
| **V4 (Elite DART - No xG)** | {mae_elite_v4:.4f} |
| **V7 (Underlying Metrics Engine)** | {mae_elite_v7:.4f} |

**Verdict:** {"V7 Successfully learned from Expected Goals!" if mae_elite_v7 < mae_elite_v4 else "V4 remains the champion. Your dataset's proxy xG might need official data to improve."}
"""
display(Markdown(report))

print("\n--- V7 Top 15 Most Accurate Elite Predictions ---")
print("Notice how the 'xGI (Form)' value influences the V7 Prediction compared to V4.")
display(v7_truth_df.sort_values(by='V7_Error', ascending=True).head(15))

In [ ]:
# %% [Cell 4: Strategy A vs B — GW29 Forecast + GW24-28 Backtest Comparison]
# This cell runs both strategies and produces:
#   1. A side-by-side GW29 prediction table
#   2. A GW24-28 backtest table with actual points and a winner column
# ── SHARED CONSTANTS ────────────────────────────────────────────────────────
_pos_to_num_shared = {'GK': 1, 'DEF': 2, 'MID': 3, 'FWD': 4, 1:1, 2:2, 3:3, 4:4}
pos_map_model_shared = {'GK': 0, 'DEF': 1, 'MID': 2, 'FWD': 3}
PENALTY_TAKERS = {'M.Salah', 'Haaland', 'Palmer', 'Saka', 'B.Fernandes',
                  'Gordon', 'Solanke', 'Jimenez'}
FIXTURE_WEIGHT = 0.3
BUDGET_CAP = 100.0

# ── HELPER: STRICT FPL AUTO-SUB + VICE-CAPTAIN LOGIC ────────────────────────
def _calculate_actual_pts(starters_df, bench_df, captain, vice_captain, target_rows):
    """
    Applies proper FPL auto-sub rules and vice-captain armband logic.
    Returns actual points scored by the team, the active captain, and their points.
    """
    if target_rows.empty or 'Total Points' not in target_rows.columns:
        return None, captain, 0

    pts_map  = target_rows.set_index('Web Name')['Total Points'].to_dict()
    mins_map = target_rows.set_index('Web Name')['Minutes Played'].to_dict() \
               if 'Minutes Played' in target_rows.columns else {}

    active_lineup = starters_df.copy()
    active_bench  = bench_df.copy()

    if mins_map:
        active_lineup['Mins'] = active_lineup['Web Name'].map(lambda x: mins_map.get(x, 0))
        active_bench['Mins']  = active_bench['Web Name'].map(lambda x: mins_map.get(x, 0))

        dnp = active_lineup[active_lineup['Mins'] == 0]
        for idx, player in dnp.iterrows():
            pos = player['Pos_ID']
            if pos == 1:
                bench_gk = active_bench[(active_bench['Pos_ID'] == 1) & (active_bench['Mins'] > 0)]
                if not bench_gk.empty:
                    active_lineup.loc[idx] = bench_gk.iloc[0]
                    active_bench = active_bench.drop(bench_gk.index[0])
            else:
                avail = active_bench[(active_bench['Pos_ID'] != 1) & (active_bench['Mins'] > 0)]
                for b_idx, sub in avail.iterrows():
                    temp = active_lineup.copy()
                    temp.loc[idx] = sub
                    if (temp['Pos_ID'] == 2).sum() >= 3 and (temp['Pos_ID'] == 4).sum() >= 1:
                        active_lineup.loc[idx] = sub
                        active_bench = active_bench.drop(b_idx)
                        break

        # Vice-captain armband logic
        cap_mins = mins_map.get(captain, 0)
        vc_mins = mins_map.get(vice_captain, 0)
        
        active_captain = captain
        if cap_mins == 0:
            if vc_mins > 0:
                active_captain = vice_captain
            else:
                active_captain = None
    else:
        active_captain = captain

    total = sum(
        pts_map.get(row['Web Name'], 0) * (2 if row['Web Name'] == active_captain else 1)
        for _, row in active_lineup.iterrows()
    )
    cap_pts = pts_map.get(active_captain, 0) if active_captain else 0
    return total, active_captain, cap_pts

# ── HELPER: build inference snapshot for any GW ─────────────────────────────
def _build_inference_snapshot(target_gw):
    hist_snap = df_augmented[
        (df_augmented['season'] == CURRENT_SEASON) &
        (df_augmented['Gameweek'] < target_gw)
    ].copy()

    target_rows = df_augmented[
        (df_augmented['season'] == CURRENT_SEASON) &
        (df_augmented['Gameweek'] == target_gw)
    ].copy()
    
    if target_rows.empty:
        target_rows = df[
            (df['season'] == CURRENT_SEASON) &
            (df['Gameweek'] == target_gw)
        ].copy()

    _is_v7 = 'V7' in globals().get('best_model_name', '')

    inf_list, names = [], []
    for pid, group in hist_snap.groupby('Player UUID'):
        if len(group) < 6:
            continue
            
        match = target_rows[target_rows['Player UUID'] == pid]
        
        if match.empty:
            if target_gw >= globals().get('CURRENT_GW', 29):
                r = group.iloc[-1]
            else:
                continue
        else:
            r = match.iloc[0]
         
        row = list(group.sort_values('Gameweek')['Total Points'].tail(6).values) + [
            pos_map_model_shared.get(r.get('Position', 'MID'), 1),
            r.get('Next_Opponent_Difficulty', 3),
            r.get('Next_Is_Home', 1),
            r.get('Rolling_Avg_Minutes_5', 60),
            r.get('Season_Phase', 3),
            r.get('Prev_Season_Avg_Points', 0),
            r.get('Avg_Pts_vs_Difficulty', 0)
        ]

        # Add rolling_xgi as 14th feature if V7 is the winning model
        if _is_v7:
            row.append(r.get('Rolling_xGI_5', 0))

        inf_list.append(row)
        names.append(r['Web Name'])

    # Build feature column list to match what was added above
    feat_cols = [f'lag_{i}' for i in range(6, 0, -1)] + [
        'position', 'next_difficulty', 'next_is_home',
        'rolling_min', 'season_phase', 'season_baseline', 'avg_pts_vs_diff'
    ]
    if _is_v7:
        feat_cols.append('rolling_xgi')

    if not inf_list:
        return pd.DataFrame(), [], {}, target_rows

    inf_df = pd.DataFrame(inf_list, columns=feat_cols)
    inf_df['position'] = inf_df['position'].astype('category')

    actual_pts_dict = {}
    if not target_rows.empty and 'Total Points' in target_rows.columns:
        actual_pts_dict = (
            df[(df['season'] == CURRENT_SEASON) & (df['Gameweek'] == target_gw)]
            .groupby('Web Name')['Total Points'].sum()
            .to_dict()
        )

    return inf_df, names, actual_pts_dict, target_rows


# ── HELPER: Strategy A squad selection ──────────────────────────────────────
def _run_strategy_a(pool_df, target_rows, inf_df):
    pool = pool_df.copy()
    pool['Weighted_xP'] = np.where(
        pool['Web Name'].isin(PENALTY_TAKERS),
        pool['Pred'] * 1.15,
        pool['Pred']
    )

    squad, p_c, t_c = [], {1:0,2:0,3:0,4:0}, {}
    limits = {1:2, 2:5, 3:5, 4:3}
    for _, p in pool.sort_values('Weighted_xP', ascending=False).iterrows():
        if p_c[p['Pos_ID']] < limits[p['Pos_ID']] and t_c.get(p['Team'], 0) < 3:
            squad.append(p)
            p_c[p['Pos_ID']] += 1
            t_c[p['Team']] = t_c.get(p['Team'], 0) + 1
        if len(squad) == 15:
            break

    df_sq = pd.DataFrame(squad)

    cheapest_map = {i: pool[pool['Pos_ID'] == i].sort_values('Price').iloc[0]['Price'] if not pool[pool['Pos_ID'] == i].empty else 4.0 for i in range(1, 5)}
    
    while df_sq['Price'].sum() > BUDGET_CAP:
        df_sq['Relief'] = (df_sq['Price'] - df_sq['Pos_ID'].map(cheapest_map)) / (df_sq['Weighted_xP'] + 0.1)
        drop_idx = df_sq['Relief'].idxmax()
        to_drop = df_sq.loc[drop_idx]
        repl = pool[
            (pool['Pos_ID'] == to_drop['Pos_ID']) &
            (~pool['Web Name'].isin(df_sq['Web Name'])) &
            (pool['Price'] < to_drop['Price'])
        ].sort_values('Weighted_xP', ascending=False).head(1)
        if not repl.empty:
            df_sq.loc[drop_idx] = repl.iloc[0]
        else:
            break

    gk    = df_sq[df_sq['Pos_ID'] == 1].nlargest(1, 'Weighted_xP')
    defs  = df_sq[df_sq['Pos_ID'] == 2].nlargest(3, 'Weighted_xP')
    fwds  = df_sq[df_sq['Pos_ID'] == 4].nlargest(1, 'Weighted_xP')
    picked = pd.concat([gk, defs, fwds])
    others = df_sq[
        ~df_sq['Web Name'].isin(picked['Web Name']) &
        (df_sq['Pos_ID'] != 1)
    ].nlargest(6, 'Weighted_xP')
    starters = pd.concat([gk, defs, fwds, others])

    if 'haul_clf' in globals():
        HAUL_THRESHOLD = globals().get('HAUL_THRESHOLD', 10)
        snap_df_haul = inf_df.copy()
        snap_df_haul['recent_haul_flag'] = (snap_df_haul[['lag_1', 'lag_2', 'lag_3']] >= HAUL_THRESHOLD).any(axis=1).astype(int)
        snap_df_haul['lag_max_6'] = snap_df_haul[[f'lag_{i}' for i in range(1, 7)]].max(axis=1)
        snap_df_haul['fixture_ease'] = 6 - snap_df_haul['next_difficulty']
        elite_names = df_augmented[df_augmented['avg_season_pts'] > 4.5]['Web Name'].unique()
        snap_df_haul['is_elite'] = [1 if n in elite_names else 0 for n in pool_df['Web Name']]
        snap_df_haul['avg_pts_vs_diff'] = inf_df['avg_pts_vs_diff'].values
        snap_df_haul['ceiling_ratio'] = snap_df_haul['lag_max_6'] / HAUL_THRESHOLD
        haul_feature_cols_a = [f'lag_{i}' for i in range(1, 7)] + [
            'position', 'next_difficulty', 'next_is_home', 'rolling_min',
            'season_phase', 'season_baseline', 'recent_haul_flag',
            'lag_max_6', 'is_elite', 'fixture_ease', 'avg_pts_vs_diff',
            'ceiling_ratio'
        ]
        haul_map = dict(zip(pool_df['Web Name'], haul_clf.predict_proba(snap_df_haul[haul_feature_cols_a])[:, 1]))
    else:
        haul_map = {}

    starters['Haul_Prob (%)'] = starters['Web Name'].map(haul_map).fillna(0.0) * 100.0
    starters['AI_Blended_Score'] = starters['Weighted_xP'] * (1 + (starters['Haul_Prob (%)'] / 100.0))
    cap_df_a = starters.sort_values(by='AI_Blended_Score', ascending=False)
    captain = cap_df_a.iloc[0]['Web Name']
    vice_captain = cap_df_a.iloc[1]['Web Name']

    pred_xp = starters['Weighted_xP'].sum() + starters[starters['Web Name'] == captain]['Weighted_xP'].values[0]
    bench = df_sq[~df_sq['Web Name'].isin(starters['Web Name'])].copy()
    actual_pts, active_cap, cap_pts = _calculate_actual_pts(starters, bench, captain, vice_captain, target_rows)

    return starters, active_cap, round(pred_xp, 1), actual_pts


# ── HELPER: Strategy B squad selection ──────────────────────────────────────
def _run_strategy_b(pool_df, target_rows, inf_df):
    pool = pool_df.copy()
    pool['Weighted_xP'] = np.where(
        pool['Web Name'].isin(PENALTY_TAKERS),
        pool['Pred'] * 1.15,
        pool['Pred']
    )

    dead_fodder = pd.concat([
        pool[pool['Pos_ID'] == 1].nsmallest(1, 'Price'),
        pool[pool['Pos_ID'] == 2].nsmallest(1, 'Price'),
        pool[pool['Pos_ID'] == 3].nsmallest(1, 'Price')
    ])

    market = pool[~pool['Web Name'].isin(dead_fodder['Web Name'])]
    core, p_c, t_c = [], {1:1, 2:4, 3:4, 4:3}, {}
    for t in dead_fodder['Team']:
        t_c[t] = t_c.get(t, 0) + 1

    for _, p in market.sort_values('Weighted_xP', ascending=False).iterrows():
        if p_c[p['Pos_ID']] > 0 and t_c.get(p['Team'], 0) < 3:
            core.append(p)
            p_c[p['Pos_ID']] -= 1
            t_c[p['Team']] = t_c.get(p['Team'], 0) + 1
        if len(core) == 12:
            break

    df_sq = pd.concat([dead_fodder, pd.DataFrame(core)]).reset_index(drop=True)
    if 'Role' not in df_sq.columns:
        df_sq['Role'] = 'Core'
    df_sq.loc[df_sq['Web Name'].isin(dead_fodder['Web Name']), 'Role'] = 'Dead Fodder'

    cheapest_map = {i: pool[pool['Pos_ID'] == i].sort_values('Price').iloc[0]['Price'] if not pool[pool['Pos_ID'] == i].empty else 4.0 for i in range(1, 5)}
    
    while df_sq['Price'].sum() > BUDGET_CAP:
        eligible = df_sq[df_sq['Role'] != 'Dead Fodder'].copy()
        eligible['Relief'] = (eligible['Price'] - eligible['Pos_ID'].map(cheapest_map)) / (eligible['Weighted_xP'] + 0.1)
        drop_idx = eligible.sort_values('Relief', ascending=False).index[0]
        to_drop = df_sq.loc[drop_idx]
        repl = pool[
            (pool['Pos_ID'] == to_drop['Pos_ID']) &
            (~pool['Web Name'].isin(df_sq['Web Name'])) &
            (pool['Price'] < to_drop['Price'])
        ].sort_values('Weighted_xP', ascending=False).head(1)
        
        if not repl.empty:
            df_sq.loc[drop_idx] = repl.iloc[0]
            df_sq.loc[drop_idx, 'Role'] = 'Core'
        else:
            break

    gk    = df_sq[df_sq['Pos_ID'] == 1].nlargest(1, 'Weighted_xP')
    defs  = df_sq[df_sq['Pos_ID'] == 2].nlargest(3, 'Weighted_xP')
    fwds  = df_sq[df_sq['Pos_ID'] == 4].nlargest(1, 'Weighted_xP')
    picked = pd.concat([gk, defs, fwds])
    others = df_sq[
        ~df_sq['Web Name'].isin(picked['Web Name']) &
        (df_sq['Pos_ID'] != 1)
    ].nlargest(6, 'Weighted_xP')
    starters = pd.concat([gk, defs, fwds, others])

    if 'haul_clf' in globals():
        HAUL_THRESHOLD = globals().get('HAUL_THRESHOLD', 10)
        snap_df_haul = inf_df.copy()
        snap_df_haul['recent_haul_flag'] = (snap_df_haul[['lag_1', 'lag_2', 'lag_3']] >= HAUL_THRESHOLD).any(axis=1).astype(int)
        snap_df_haul['lag_max_6'] = snap_df_haul[[f'lag_{i}' for i in range(1, 7)]].max(axis=1)
        snap_df_haul['fixture_ease'] = 6 - snap_df_haul['next_difficulty']
        elite_names = df_augmented[df_augmented['avg_season_pts'] > 4.5]['Web Name'].unique()
        snap_df_haul['is_elite'] = [1 if n in elite_names else 0 for n in pool_df['Web Name']]
        snap_df_haul['avg_pts_vs_diff'] = inf_df['avg_pts_vs_diff'].values
        snap_df_haul['ceiling_ratio'] = snap_df_haul['lag_max_6'] / HAUL_THRESHOLD
        haul_feature_cols_b = [f'lag_{i}' for i in range(1, 7)] + [
            'position', 'next_difficulty', 'next_is_home', 'rolling_min',
            'season_phase', 'season_baseline', 'recent_haul_flag',
            'lag_max_6', 'is_elite', 'fixture_ease', 'avg_pts_vs_diff',
            'ceiling_ratio'
        ]
        haul_map = dict(zip(pool_df['Web Name'], haul_clf.predict_proba(snap_df_haul[haul_feature_cols_b])[:, 1]))
    else:
        haul_map = {}

    starters['Haul_Prob (%)'] = starters['Web Name'].map(haul_map).fillna(0.0) * 100.0
    starters['AI_Blended_Score'] = starters['Weighted_xP'] * (1 + (starters['Haul_Prob (%)'] / 100.0))
    cap_df_b = starters.sort_values(by='AI_Blended_Score', ascending=False)
    captain = cap_df_b.iloc[0]['Web Name']
    vice_captain = cap_df_b.iloc[1]['Web Name']

    pred_xp = starters['Weighted_xP'].sum() + starters[starters['Web Name'] == captain]['Weighted_xP'].values[0]
    bench = df_sq[~df_sq['Web Name'].isin(starters['Web Name'])].copy()
    actual_pts, active_cap, cap_pts = _calculate_actual_pts(starters, bench, captain, vice_captain, target_rows)

    return starters, active_cap, round(pred_xp, 1), actual_pts

# ── HELPER: build pool_df from inference snapshot ────────────────────────────
def _build_pool(inf_df, names):
    if inf_df.empty:
        return pd.DataFrame()

    # Lock categorical position to match training categories
    if 'X_train_raw' in globals() and 'position' in X_train_raw.columns:
        inf_df = inf_df.copy()
        inf_df['position'] = pd.Categorical(
            inf_df['position'],
            categories=X_train_raw['position'].cat.categories
        )

    pool = pd.DataFrame({'Web Name': names, 'Pred': winning_model.predict(inf_df)})
    pool['Price']  = pool['Web Name'].map(lambda x: live_data.get(x, {}).get('price', 5.5))
    pool['Pos_ID'] = pool['Web Name'].map(pos_map).map(_pos_to_num_shared).fillna(3).astype(int)
    pool['Team']   = pool['Web Name'].map(club_map).fillna('Neutral')
    return pool



# ════════════════════════════════════════════════════════════════════════════
# SECTION 1 — GW29 LIVE FORECAST (side-by-side)
# ════════════════════════════════════════════════════════════════════════════
CURRENT_GW = globals().get('CURRENT_GW', 29)

print("=" * 70)
print(f"  GW{CURRENT_GW} LIVE FORECAST — STRATEGY A vs STRATEGY B")
print("=" * 70)

inf_df_live, names_live, _, _ = _build_inference_snapshot(CURRENT_GW)
pool_live = _build_pool(inf_df_live, names_live)

target_rows_live = df_augmented[
    (df_augmented['season'] == CURRENT_SEASON) &
    (df_augmented['Gameweek'] == CURRENT_GW - 1)
].copy()

if not pool_live.empty:
    starters_a, cap_a, pred_a, _ = _run_strategy_a(pool_live, target_rows_live, inf_df_live)
    starters_b, cap_b, pred_b, _ = _run_strategy_b(pool_live, target_rows_live, inf_df_live)

    print(f"\n{'Player':<22} {'Strategy A xP':>14} {'Strategy B xP':>14}")
    print("-" * 52)

    all_names = sorted(set(starters_a['Web Name'].tolist()) | set(starters_b['Web Name'].tolist()))
    for name in all_names:
        in_a = starters_a[starters_a['Web Name'] == name]
        in_b = starters_b[starters_b['Web Name'] == name]
        xp_a_str = f"{in_a['Weighted_xP'].values[0]:.1f}" if not in_a.empty else "—"
        xp_b_str = f"{in_b['Weighted_xP'].values[0]:.1f}" if not in_b.empty else "—"
        cap_flag_a = " (C)" if name == cap_a else ""
        cap_flag_b = " (C)" if name == cap_b else ""
        print(f"{name:<22} {xp_a_str+cap_flag_a:>14} {xp_b_str+cap_flag_b:>14}")

    print("-" * 52)
    print(f"{'TOTAL xP':<22} {pred_a:>14.1f} {pred_b:>14.1f}")
    print(f"{'CAPTAIN':<22} {cap_a:>14} {cap_b:>14}")
else:
    print("Could not generate live forecast. Ensure historical data exists.")


# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 — GW24-28 BACKTEST COMPARISON TABLE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  GW24-28 BACKTEST — STRATEGY A vs STRATEGY B")
print("=" * 70)

backtest_results = []

for gw in range(24, 29):
    inf_df_gw, names_gw, _, target_rows_gw = _build_inference_snapshot(gw)
    if inf_df_gw.empty:
        print(f"  GW{gw}: no data, skipping.")
        continue

    pool_gw = _build_pool(inf_df_gw, names_gw)
    
    if pool_gw.empty:
        continue

    _, cap_a_gw, pred_a_gw, actual_a_gw = _run_strategy_a(pool_gw, target_rows_gw, inf_df_gw)
    _, cap_b_gw, pred_b_gw, actual_b_gw = _run_strategy_b(pool_gw, target_rows_gw, inf_df_gw)

    if actual_a_gw is not None and actual_b_gw is not None:
        if actual_a_gw > actual_b_gw:
            winner = "Strategy A"
        elif actual_b_gw > actual_a_gw:
            winner = "Strategy B"
        else:
            winner = "Draw"
    else:
        winner = "N/A"

    backtest_results.append({
        'GW':                  gw,
        'A Predicted':         pred_a_gw,
        'A Actual':            actual_a_gw if actual_a_gw is not None else '—',
        'A Captain':           cap_a_gw,
        'B Predicted':         pred_b_gw,
        'B Actual':            actual_b_gw if actual_b_gw is not None else '—',
        'B Captain':           cap_b_gw,
        'Winner (by Actual)':  winner
    })

if backtest_results:
    backtest_df = pd.DataFrame(backtest_results)
    display_cols = ['GW', 'A Predicted', 'A Actual', 'A Captain', 'B Predicted', 'B Actual', 'B Captain', 'Winner (by Actual)']
    display(backtest_df[display_cols])

    numeric_rows = backtest_df[backtest_df['A Actual'].apply(lambda x: isinstance(x, (int, float)))]
    total_a = numeric_rows['A Actual'].sum()
    total_b = numeric_rows['B Actual'].sum()
    wins_a  = (backtest_df['Winner (by Actual)'] == 'Strategy A').sum()
    wins_b  = (backtest_df['Winner (by Actual)'] == 'Strategy B').sum()
    draws   = (backtest_df['Winner (by Actual)'] == 'Draw').sum()

    print("\n--- EXECUTIVE SUMMARY (GW24-28) ---")
    print(f"Strategy A  |  Total Actual Pts: {total_a:.0f}  |  GW Wins: {wins_a}")
    print(f"Strategy B  |  Total Actual Pts: {total_b:.0f}  |  GW Wins: {wins_b}")
    if draws:
        print(f"Draws: {draws}")
    print("-" * 45)
    if total_a > total_b:
        print(f"OVERALL WINNER: Strategy A (+{total_a - total_b:.0f} pts over 5 GWs)")
    elif total_b > total_a:
        print(f"OVERALL WINNER: Strategy B (+{total_b - total_a:.0f} pts over 5 GWs)")
    else:
        print("OVERALL RESULT: Draw")
else:
    print("Could not generate backtest. Check source data.")

In [ ]:
# %% [Cell: Master Squad Optimizer - Strategy B (V7 EXPERIMENTAL ENGINE)]
# This is an identical clone of Strategy B, hard-wired to use V7 (xG/xA) logic.

def solve_wildcard_v7_experimental(budget_cap=100.0):
    # --- 1. DATA PREP (V7 INFERENCE) ---
    pos_to_num = {'GK': 1, 'DEF': 2, 'MID': 3, 'FWD': 4, 1:1, 2:2, 3:3, 4:4}
    
    # We build the leaderboard using V7 logic instead of reading final_leaderboard
    inference_v7 = []
    names_v7 = []
    active_players = df_augmented[df_augmented['season'] == CURRENT_SEASON].copy()
    
    # V7 must have these 13 features in this exact order
    v7_feat_cols = [f'lag_{i}' for i in range(6, 0, -1)] + [
        'position', 'next_difficulty', 'next_is_home', 'rolling_min',
        'season_phase', 'season_baseline', 'avg_pts_vs_diff', 'rolling_xgi'
    ]

    for pid, group in active_players.groupby('Player UUID'):
        group = group.sort_values('Gameweek')
        if len(group) >= 6:
            lags = list(group['Total Points'].tail(6).values)
            last = group.iloc[-1]
            row = lags + [
                last['Position'], last['Next_Opponent_Difficulty'], last['Next_Is_Home'], 
                last['Rolling_Avg_Minutes_5'], last['Season_Phase'], last['Prev_Season_Avg_Points'],
                last['Avg_Pts_vs_Difficulty'], last['Rolling_xGI_5']
            ]
            inference_v7.append(row)
            names_v7.append(last['Web Name'])

    inf_df_v7 = pd.DataFrame(inference_v7, columns=v7_feat_cols)
    inf_df_v7['position'] = inf_df_v7['position'].astype('category')
    
    # Generate V7 Leaderboard
    v7_preds = model_v7.predict(inf_df_v7)
    leaderboard = pd.DataFrame({'Player': names_v7, 'Predicted_Points': v7_preds})

    # Apply your standardized mapping logic
    _global_pos_map = globals().get('pos_map', {})
    _global_club_map = globals().get('club_map', {})

    leaderboard['Price']  = leaderboard['Player'].map(lambda x: live_data.get(x, {}).get('price', 5.5))
    leaderboard['Pos_ID'] = leaderboard['Player'].map(lambda x: pos_to_num.get(_global_pos_map.get(x, 'MID'), 3))
    leaderboard['Team']   = leaderboard['Player'].map(_global_club_map).fillna('Neutral')
    leaderboard['Status'] = leaderboard['Player'].map(lambda x: live_data.get(x, {}).get('status', 'a'))
    leaderboard['Chance'] = leaderboard['Player'].map(
        lambda x: live_data.get(x, {}).get('chance_of_playing_this_gw', 100) or 100
    )

    # --- 2. AVAILABILITY FILTER (Synced with global squad_availability) ---
    available_pool = leaderboard.copy()
    _av = globals().get('squad_availability')
    
    if isinstance(_av, dict):
        _sell_now = _av.get('sell_now', [])
        _doubtful = _av.get('doubtful', [])
        available_pool = available_pool[~available_pool['Player'].isin(_sell_now)]
        available_pool['Predicted_Points'] = np.where(
            available_pool['Player'].isin(_doubtful),
            available_pool['Predicted_Points'] - 10,
            available_pool['Predicted_Points']
        )
    else:
        available_pool = leaderboard[
            (leaderboard['Status'] == 'a') |
            ((leaderboard['Status'] == 'd') & (leaderboard['Chance'] >= 50))
        ].copy()

    # --- 3. PENALTY TAKER WEIGHTING ---
    penalty_takers = {'M.Salah', 'Haaland', 'Palmer', 'Saka', 'B.Fernandes', 'Gordon', 'Solanke', 'Jimenez'}
    available_pool['Weighted_xP'] = np.where(
        available_pool['Player'].isin(penalty_takers),
        available_pool['Predicted_Points'] * 1.15,
        available_pool['Predicted_Points']
    )

    pool = available_pool.sort_values('Weighted_xP', ascending=False)

    # --- 4. THE 3 DEAD SLOTS (cheapest available per position) ---
    dead_fodder = pd.concat([
        pool[pool['Pos_ID'] == 1].nsmallest(1, 'Price'),
        pool[pool['Pos_ID'] == 2].nsmallest(1, 'Price'),
        pool[pool['Pos_ID'] == 3].nsmallest(1, 'Price')
    ])
    dead_fodder = dead_fodder.copy()
    dead_fodder['Role'] = 'Dead Fodder'

    # --- 5. CORE 12 SELECTION (11 starters + 1 safety sub) ---
    market = pool[~pool['Player'].isin(dead_fodder['Player'])]
    core_squad_list = []
    team_counts = dead_fodder['Team'].value_counts().to_dict()
    pos_needed  = {1: 1, 2: 4, 3: 4, 4: 3}

    for _, player in market.iterrows():
        pid, team = player['Pos_ID'], player['Team']
        if pos_needed[pid] > 0 and team_counts.get(team, 0) < 3:
            core_squad_list.append(player)
            pos_needed[pid] -= 1
            team_counts[team] = team_counts.get(team, 0) + 1
        if len(core_squad_list) == 12:
            break

    squad = pd.concat([dead_fodder, pd.DataFrame(core_squad_list)]).reset_index(drop=True)
    if 'Role' not in squad.columns:
        squad['Role'] = 'Core'
    squad.loc[squad['Player'].isin(dead_fodder['Player']), 'Role'] = 'Dead Fodder'

    # --- 6. BUDGET REPAIR ---
    while squad['Price'].sum() > budget_cap:
        eligible = squad[squad['Role'] != 'Dead Fodder'].copy()
        eligible['Efficiency'] = eligible['Weighted_xP'] / (eligible['Price'] + 0.1)
        drop_idx  = eligible.sort_values('Efficiency').index[0]
        p_to_drop = squad.loc[drop_idx]

        replacement = pool[
            (pool['Pos_ID'] == p_to_drop['Pos_ID']) &
            (~pool['Player'].isin(squad['Player'])) &
            (pool['Price'] < p_to_drop['Price'])
        ].sort_values('Predicted_Points', ascending=False).head(1)

        if replacement.empty:
            break
        squad.loc[drop_idx] = replacement.iloc[0]
        squad.loc[drop_idx, 'Role'] = 'Core'

    # --- 7. STARTING 11 SELECTION ---
    gk   = squad[squad['Pos_ID'] == 1].nlargest(1, 'Weighted_xP')
    defs = squad[squad['Pos_ID'] == 2].nlargest(3, 'Weighted_xP')
    fwds = squad[squad['Pos_ID'] == 4].nlargest(1, 'Weighted_xP')

    already_ids  = pd.concat([gk, defs, fwds])['Player'].tolist()
    outfield_rem = squad[(~squad['Player'].isin(already_ids)) & (squad['Pos_ID'] != 1)]
    others       = outfield_rem.nlargest(6, 'Weighted_xP')

    opt_11 = pd.concat([gk, defs, fwds, others]).copy()

    # --- 8. DUAL-ENGINE CAPTAINCY ---
    haul_map = hauler_board.set_index('Player')['Haul_Prob (%)'].to_dict()
    opt_11['Haul_Prob (%)']    = opt_11['Player'].map(haul_map).fillna(0.0)
    opt_11['AI_Blended_Score'] = opt_11['Weighted_xP'] * (1 + (opt_11['Haul_Prob (%)'] / 100.0))

    cap_df       = opt_11.sort_values(by='AI_Blended_Score', ascending=False)
    captain_name = cap_df.iloc[0]['Player']
    vice_name    = cap_df.iloc[1]['Player']

    opt_11['Final_Weighted_xP'] = opt_11['Weighted_xP']
    opt_11.loc[opt_11['Player'] == captain_name, 'Final_Weighted_xP'] *= 2
    opt_bench = squad[~squad['Player'].isin(opt_11['Player'])]

    # --- 9. OUTPUT ---
    print(f"--- 🧪 STRATEGY B: V7 EXPERIMENTAL ENGINE (GW {CURRENT_GW}) ---")
    print(f"Total Cost: £{squad['Price'].sum():.1f}m / £{budget_cap:.1f}m")
    print(f"EXPECTED STARTING 11 POINTS (V7): {opt_11['Final_Weighted_xP'].sum():.2f}")

    print("\n[STARTING 11]")
    display(opt_11[['Player', 'Team', 'Weighted_xP', 'Price']])
    captain_2x = opt_11.loc[opt_11['Player'] == captain_name, 'Weighted_xP'].values[0]
    print(f"  Captain bonus: {captain_name} scores {captain_2x:.2f} xP → counts as {captain_2x*2:.2f} xP (×2)")
    print(f"  Total xP including captain double: {opt_11['Weighted_xP'].sum() + captain_2x:.2f}")

    print("\n[BENCH]")
    display(opt_bench[['Player', 'Team', 'Weighted_xP', 'Price']])

    print("\n--- DUAL-ENGINE CAPTAIN RANKING ---")
    display_cap = cap_df[['Player', 'Weighted_xP', 'Haul_Prob (%)', 'AI_Blended_Score']].head(5).copy()
    display_cap['AI_Blended_Score'] = display_cap['AI_Blended_Score'].round(2)
    display(display_cap)
    print("-" * 50)
    print(f"CAPTAIN:      {captain_name} (Blended Score: {cap_df.iloc[0]['AI_Blended_Score']:.2f})")
    print(f"VICE-CAPTAIN: {vice_name} (Blended Score: {cap_df.iloc[1]['AI_Blended_Score']:.2f})")
    print("-" * 50)

    return opt_11, opt_bench, squad['Price'].sum()

# Run the experimental V7 audit
opt_11_v7, opt_bench_v7, total_spent_v7 = solve_wildcard_v7_experimental()

In [ ]:
# %% [Cell: V7 Strategy B — GW 24-28 Historical Backtest Simulation]
import pandas as pd
import numpy as np

print("--- 🔄 RUNNING V7 HISTORICAL BACKTEST (GW 24-28) ---")
print("Simulating Strategy B using Underlying Metrics (xG/xA)...")

v7_historical_results = []

for gw in range(24, 29):
    inf_df_gw, names_gw, _, target_rows_gw = _build_inference_snapshot(gw)
    if inf_df_gw.empty:
        continue
    # Build V7 features for this GW snapshot
    # ... add rolling_xgi from df_augmented for this specific GW
    pool_gw = _build_pool_v7(inf_df_gw, names_gw)
    _, cap_gw, pred_gw, actual_gw = _run_strategy_b(pool_gw, target_rows_gw, inf_df_gw)
    _opt_11, _opt_bench, _spent = solve_wildcard_v7_experimental()
    
    # 3. Get the real match results for that specific Gameweek
    target_rows = df[(df['season'] == CURRENT_SEASON) & (df['Gameweek'] == gw)]
    
    # 4. Identify captain and vice-captain from the V7 results
    _captain = _opt_11[_opt_11['Final_Weighted_xP'] > _opt_11['Weighted_xP'] * 1.5].iloc[0]['Player']
    _vice = _opt_11.sort_values('AI_Blended_Score', ascending=False).iloc[1]['Player']
    
    # 5. Calculate actual points scored using your official auto-sub logic
    actual_pts, active_cap, cap_pts = _calculate_actual_pts(
        _opt_11.rename(columns={'Player': 'Web Name'}), 
        _opt_bench.rename(columns={'Player': 'Web Name'}), 
        _captain, _vice, target_rows
    )
    
    v7_historical_results.append({
        'GW': gw,
        'V7 Predicted': round(_opt_11['Final_Weighted_xP'].sum(), 2),
        'B Actual': actual_pts, 
        'Captain': active_cap
    })

# Save to the variable the UI is looking for
v7_backtest_df = pd.DataFrame(v7_historical_results)

print("\n✅ V7 Backtest Complete!")
display(v7_backtest_df)

In [ ]:
# %% [Cell: Benchmark Comparison — V7 AI vs GW Averages]
import pandas as pd
import numpy as np
import os
from pathlib import Path
from IPython.display import display, HTML

# ── 1. LOAD BENCHMARKS ────────────────────────────────────────────────────────
_known_path = Path(r'C:\Users\skourako\diplo\fpl_pipeline\data\GW averages.xlsx')
benchmarks = None

if _known_path.exists():
    benchmarks = pd.read_excel(_known_path)
else:
    _candidate = Path(os.getcwd()) / 'data' / 'GW averages.xlsx'
    if _candidate.exists():
        benchmarks = pd.read_excel(_candidate)

if benchmarks is None:
    print(f"[ERROR] GW averages.xlsx not found.")
elif 'v7_backtest_df' not in globals() or v7_backtest_df.empty:
    print("[WARNING] v7_backtest_df not found. Run the V7 Backtest cell first.")
else:
    benchmarks.columns = ['GW', 'Average_Manager', 'Top5pct_Manager', 'Dream_Team']

    # ── 2. MERGE WITH V7 DATA ─────────────────────────────────────────────────
    # We now pull from v7_backtest_df
    comp = v7_backtest_df[['GW', 'B Actual']].copy()
    comp = comp.merge(benchmarks, on='GW', how='left')

    comp['AI vs Average (pts)']  = comp['B Actual'] - comp['Average_Manager']
    comp['AI vs Top 5% (pts)']   = comp['B Actual'] - comp['Top5pct_Manager']
    comp['Beat Average?']        = comp['AI vs Average (pts)'].apply(lambda x: 'YES' if x >= 0 else 'NO')
    comp['Beat Top 5%?']         = comp['AI vs Top 5% (pts)'].apply(lambda x: 'YES' if x >= 0 else 'NO')

    # ── 3. STYLES ─────────────────────────────────────────────────────────────
    styles = """
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700;800&display=swap');
    .bm-wrap { font-family:'Poppins',sans-serif; width:860px; margin:auto; box-shadow: 0 10px 30px rgba(0,0,0,0.1); border-radius: 12px; overflow:hidden;}
    .bm-header { background: linear-gradient(135deg,#37003c,#1a0020); padding: 16px 20px; display: flex; align-items: center; justify-content: space-between; border-bottom: 3px solid #00c2ff; }
    .bm-title { color:#00c2ff; font-weight:800; font-size:0.95em; letter-spacing:1.5px; text-transform:uppercase; }
    .bm-pill  { border-radius:20px; padding:4px 14px; font-size:0.7em; font-weight:700; background:rgba(0,194,255,0.15); border:1px solid #00c2ff; color:#00c2ff; }
    .bm-table { width:100%; border-collapse:collapse; font-size:0.85em; background:white; }
    .bm-table th { background:#f8f9fa; color:#37003c; padding:12px 10px; text-align:center; font-weight:800; border-bottom:2px solid #ddd; }
    .bm-table td { padding:12px 10px; text-align:center; border-bottom:1px solid #f0f0f0; font-weight:600; color:#333; }
    .pos-diff { font-weight:800; color:#00803e; background:#e8fff4; padding: 4px 8px; border-radius: 6px;}
    .neg-diff { font-weight:800; color:#cc0000; background:#fff0f0; padding: 4px 8px; border-radius: 6px;}
    .badge-yes { background:#00ff87; color:#003a1f; border-radius:12px; padding:4px 10px; font-size:0.8em; font-weight:800; }
    .badge-no  { background:#ff4b4b; color:#fff; border-radius:12px; padding:4px 10px; font-size:0.8em; font-weight:800; }
    .bm-summary { background: linear-gradient(135deg,#37003c,#1a0020); padding: 20px; border-top: 3px solid #00c2ff; display: flex; justify-content: space-around; }
    .bm-stat-val { font-weight:800; font-size:1.6em; color:white; }
    .bm-stat-label { color:rgba(255,255,255,0.6); font-size:0.65em; font-weight:700; text-transform:uppercase; }
    </style>
    """

    # ── 4. BUILD TABLE ROWS ───────────────────────────────────────────────────
    rows_html = ""
    for _, row in comp.iterrows():
        ai_vs_avg  = row['AI vs Average (pts)']
        ai_vs_top5 = row['AI vs Top 5% (pts)']
        rows_html += f"""
        <tr>
            <td style="font-weight:800;">GW {int(row['GW'])}</td>
            <td style="color:#37003c; font-size:1.1em;">{row['B Actual']:.0f}</td>
            <td>{row['Average_Manager']:.0f}</td>
            <td>{row['Top5pct_Manager']:.0f}</td>
            <td><span class="{'pos-diff' if ai_vs_avg >= 0 else 'neg-diff'}">{'+' if ai_vs_avg >=0 else ''}{ai_vs_avg:.0f}</span></td>
            <td><span class="{'pos-diff' if ai_vs_top5 >= 0 else 'neg-diff'}">{'+' if ai_vs_top5 >=0 else ''}{ai_vs_top5:.0f}</span></td>
            <td><span class="{'badge-yes' if ai_vs_avg >= 0 else 'badge-no'}">{'YES' if ai_vs_avg >= 0 else 'NO'}</span></td>
            <td><span class="{'badge-yes' if ai_vs_top5 >= 0 else 'badge-no'}">{'YES' if ai_vs_top5 >= 0 else 'NO'}</span></td>
        </tr>"""

    # ── 5. ASSEMBLE ───────────────────────────────────────────────────────────
    avg_ai = comp['B Actual'].mean()
    avg_top5 = comp['Top5pct_Manager'].mean()
    beat_avg_count = (comp['Beat Average?'] == 'YES').sum()
    n_gws = len(comp)

    full_html = styles + f"""
    <div class="bm-wrap">
        <div class="bm-header">
            <div class="bm-title">V7 Engine Showdown — xG Intel vs The World</div>
            <div class="bm-pill">Strategy B (V7 Engine)</div>
        </div>
        <table class="bm-table">
            <thead>
                <tr>
                    <th>GW</th><th>V7 Points</th><th>Global Avg</th><th>Top 5%</th>
                    <th>vs Global</th><th>vs Top 5%</th><th>Beat Global?</th><th>Beat Top 5%?</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
        <div class="bm-summary">
            <div style="text-align:center;">
                <div class="bm-stat-val" style="color:#00ff87;">{avg_ai:.1f}</div>
                <div class="bm-stat-label">V7 Average Pts</div>
            </div>
            <div style="text-align:center;">
                <div class="bm-stat-val" style="color:#00c2ff;">{beat_avg_count} / {n_gws}</div>
                <div class="bm-stat-label">Beat Global Average</div>
            </div>
            <div style="text-align:center;">
                <div class="bm-stat-val" style="color:#ffb703;">{avg_top5:.0f}</div>
                <div class="bm-stat-label">Top 5% Benchmark</div>
            </div>
        </div>
    </div>"""

    display(HTML(full_html))

In [ ]:
# %% [Cell: V7 Pipeline — Underlying Metrics Engine (xG/xA)]
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

print("--- BUILDING V7 (xG/xA) TRAINING WINDOWS ---")

def generate_fpl_windows_v7(df, window_size=6):
    train_data = []
    source_indices = []
    df = df.sort_values(['Player UUID', 'season', 'Gameweek'])

    for pid, group in df.groupby('Player UUID'):
        points = group['Total Points'].values
        if len(points) <= window_size: continue

        pos = group['Position'].iloc[0]
        next_diff = group['Next_Opponent_Difficulty'].values
        next_home = group['Next_Is_Home'].values
        avg_min   = group['Rolling_Avg_Minutes_5'].values
        phase     = group['Season_Phase'].values
        baseline  = group['Prev_Season_Avg_Points'].values
        avg_diff  = group['Avg_Pts_vs_Difficulty'].values
        xgi       = group['Rolling_xGI_5'].values
        
        group_indices = group.index.tolist()

        for i in range(len(points) - window_size):
            t_idx = i + window_size - 1
            target_points = points[i + window_size]
            
            if np.all(points[i : i + window_size] == 0) and target_points == 0: continue

            row = list(points[i : i + window_size]) + [
                pos, next_diff[t_idx], next_home[t_idx], avg_min[t_idx],
                phase[t_idx], baseline[t_idx], avg_diff[t_idx],
                xgi[t_idx], target_points
            ]
            train_data.append(row)
            source_indices.append(group_indices[i + window_size])

    cols = [f'lag_{i}' for i in range(window_size, 0, -1)] + [
            'position', 'next_difficulty', 'next_is_home', 'rolling_min',
            'season_phase', 'season_baseline', 'avg_pts_vs_diff', 
            'rolling_xgi', 'target']
    
    result_df = pd.DataFrame(train_data, columns=cols)
    result_df.index = source_indices 
    result_df['position'] = result_df['position'].astype('category')
    return result_df

df_win_v7 = generate_fpl_windows_v7(df_augmented)

# 2. Build a dynamic, crash-proof Validation Mask
VAL_SEASON = "2025-26"
VAL_START_GW = 24
VAL_END_GW = 28

source_meta_v7 = df_augmented.loc[df_win_v7.index, ['season', 'Gameweek']]
v7_val_mask = (
    (source_meta_v7['season'] == VAL_SEASON) &
    (source_meta_v7['Gameweek'] >= VAL_START_GW) &
    (source_meta_v7['Gameweek'] <= VAL_END_GW)
).values

X_v7_all = df_win_v7.drop(columns=['target'])
y_v7_all = df_win_v7['target']

X_train_v7 = X_v7_all[~v7_val_mask].copy()
X_val_v7   = X_v7_all[v7_val_mask].copy()
y_train_v7 = y_v7_all[~v7_val_mask].copy()
y_val_v7   = y_v7_all[v7_val_mask].copy()

# 3. Train the V7 DART Model
# DYNAMIC ELITE MASK: Recalculate here to avoid Jupyter state size mismatches
player_season_avgs = df_augmented.groupby('Player UUID')['Total Points'].mean()
elite_uuids = player_season_avgs[player_season_avgs > 4.5].index

window_uuids = df_augmented.loc[df_win_v7.index, 'Player UUID']
elite_weights_mask = window_uuids.isin(elite_uuids).values
train_weights_v7 = np.where(elite_weights_mask[~v7_val_mask], 2.5, 1.0)

print(f"Training V7 Underlying Metrics Engine on {len(X_train_v7):,} windows...")

model_v7 = lgb.LGBMRegressor(
    objective='regression', boosting_type='dart', n_estimators=1200, 
    learning_rate=0.05, drop_rate=0.1, reg_alpha=0.5, reg_lambda=0.5, 
    max_depth=7, verbosity=-1, random_state=42
)

model_v7.fit(
    X_train_v7, y_train_v7,
    sample_weight=train_weights_v7,
    eval_set=[(X_val_v7, y_val_v7)],
    categorical_feature=['position', 'next_is_home'],
    callbacks=[lgb.log_evaluation(period=200)]
)

mae_v7 = mean_absolute_error(y_val_v7, model_v7.predict(X_val_v7))
print(f"\n✅ V7 Validation MAE: {mae_v7:.4f}")

In [ ]:
# %% [Cell: V7 Data Enrichment - Expected Goals (xG) & Assists (xA)]
import pandas as pd
import numpy as np

print("--- INJECTING UNDERLYING METRICS (xG & xA) ---")

# 1. Safely locate the xG/xA columns directly in df_augmented
xg_col = next((c for c in df_augmented.columns if 'expected_goals' in c.lower() or 'xg' in c.lower()), None)
xa_col = next((c for c in df_augmented.columns if 'expected_assists' in c.lower() or 'xa' in c.lower()), None)

# Build Mathematical Proxy Features if official ones are missing
if not xg_col or not xa_col:
    print("[NOTE] Official xG/xA columns not found. Building Mathematical Proxy Features...")
    threat_col = next((c for c in df_augmented.columns if 'threat' in c.lower()), None)
    create_col = next((c for c in df_augmented.columns if 'creativity' in c.lower()), None)
    
    if threat_col and create_col:
        df_augmented['xG_metric'] = df_augmented[threat_col] / 100.0  
        df_augmented['xA_metric'] = df_augmented[create_col] / 100.0
    else:
        df_augmented['xG_metric'] = df_augmented.get('goals_scored', df_augmented.get('Goals', 0)) * 0.85
        df_augmented['xA_metric'] = df_augmented.get('assists', df_augmented.get('Assists', 0)) * 0.85
    
    xg_col, xa_col = 'xG_metric', 'xA_metric'
else:
    print(f"Found official underlying metrics: '{xg_col}' and '{xa_col}'")

# 2. Sort chronologically to prevent data leakage
df_augmented = df_augmented.sort_values(['Player UUID', 'season', 'Gameweek'])

# 3. Calculate 5-Match Rolling Averages (Lagged by 1 so it only uses PAST data)
df_augmented['Rolling_xG_5'] = df_augmented.groupby('Player UUID')[xg_col].transform(
    lambda x: pd.to_numeric(x, errors='coerce').shift(1).rolling(5, min_periods=1).mean()
).fillna(0)

df_augmented['Rolling_xA_5'] = df_augmented.groupby('Player UUID')[xa_col].transform(
    lambda x: pd.to_numeric(x, errors='coerce').shift(1).rolling(5, min_periods=1).mean()
).fillna(0)

# Combine into xGI (Expected Goal Involvement)
df_augmented['Rolling_xGI_5'] = df_augmented['Rolling_xG_5'] + df_augmented['Rolling_xA_5']

print("SUCCESS: Injected 'Rolling_xGI_5' without altering row counts.")
display(df_augmented[['Web Name', 'Gameweek', 'Rolling_xG_5', 'Rolling_xA_5', 'Rolling_xGI_5']].tail(5))

In [ ]:
#  [Cell: Final Data Scrub - Removing Verified Ghosts]

# List of players verified as inactive in the 2025-26 Premier League
verified_ghosts = ['Thiago', 'Mane', 'Bale', 'Sadio Mané']

# Filter out these specific names ONLY for the 2025-26 season rows
# We keep them for older seasons so the model can still learn from their history
initial_count = len(df_augmented)

df_augmented = df_augmented[~(
    (df_augmented['Web Name'].isin(verified_ghosts)) & 
    (df_augmented['season'] == '2025-26')
)].copy()

removed_count = initial_count - len(df_augmented)
print(f"Surgical Clean Complete: Removed {removed_count} ghost rows from 2025-26.")

In [ ]:
# %% [Cell: Final Data Scrub - Removing Inactive Players & Position Fixes]

# 1. PURGE INACTIVE PLAYERS (Ensure they don't take up Dead Fodder slots)
verified_ghosts = ['Thiago', 'Mane', 'Bale', 'Sadio Mané', 'Joe Anderson', 'Karl Darlow']

initial_count = len(df_augmented)
df_augmented = df_augmented[~(
    (df_augmented['Web Name'].isin(verified_ghosts)) & 
    (df_augmented['season'] == '2025-26')
)].copy()

print(f"Purge Complete: Removed {initial_count - len(df_augmented)} rows.")

# 2. OVERRIDE MAPS (Fix Madueke and Cucurella for 2025-26 reality)
manual_overrides = {
    'Madueke':  ('MID', 'Arsenal'),
    'Cucurella': ('DEF', 'Chelsea'),
    'Anderson':  ('MID', 'Nott\'m Forest'),
    'Martinez':  ('GK',  'Aston Villa'),
    'James':     ('DEF', 'Chelsea')
}

# Safely access the global maps
_global_pos_map = globals().get('pos_map')
_global_club_map = globals().get('club_map')

if _global_pos_map is not None and _global_club_map is not None:
    for name, (pos, club) in manual_overrides.items():
        # Update if the name exists in our maps
        if name in _global_pos_map:
            _global_pos_map[name] = pos
        if name in _global_club_map:
            _global_club_map[name] = club
    print("✅ Global Position and Club maps updated successfully.")
else:
    print("⚠️ WARNING: 'pos_map' or 'club_map' not found in memory.")
    print("Run the [Collision-Safe Lookup Maps] cell first, then re-run this cell.")